# Mock Datasets

In [ ]:
# We predefine the thresholds for burnout risk levels
burnout_threshold_dict = {
    "HIGH": 0.70,
    "MEDIUM": 0.5,
}

# Since our model has RMSE, we
rmse = 0.1
stress_threshold_dict = {
    "HIGH": burnout_threshold_dict["HIGH"] - rmse,
    "MEDIUM": burnout_threshold_dict["MEDIUM"] - rmse
}

In [20]:
import pandas as pd
import numpy as np

# 1. Setup Timeline
days = np.arange(1, 31)
stress_levels = []

# 2. Generate Synthetic "User Journey"
for day in days:
    if day <= 14:
        # Phase 1: Healthy Baseline (Low Stress)
        val = np.random.uniform(0.0, 0.1)
    elif 15 <= day <= 18:
        # Phase 2: The Sudden Spiral (Acute Spike)
        val = np.random.uniform(0.85, 0.95)
    else:
        # Phase 3: Chronic Fatigue (Sustained High Stress)
        val = np.random.uniform(0.70, 0.85)
    stress_levels.append(round(val, 3))

# 3. Create DataFrame
mock_data_1 = pd.DataFrame({
    'day': days,
    'predicted_stress': stress_levels
})

# 4. Display for Verification
print(mock_data_1.head(15))

    day  predicted_stress
0     1             0.066
1     2             0.097
2     3             0.026
3     4             0.043
4     5             0.088
5     6             0.045
6     7             0.023
7     8             0.034
8     9             0.070
9    10             0.006
10   11             0.048
11   12             0.029
12   13             0.021
13   14             0.037
14   15             0.947


In [3]:
import pandas as pd
import numpy as np

# 1. Setup Timeline (30 Days)
days = np.arange(1, 31)
stress_levels = []

# 2. Generate "High-Pressure Cycle"
for day in days:
    weekday = (day - 1) % 7  # 0=Mon, 1=Tue... 5=Sat, 6=Sun
    
    if day <= 14:
        # Phase 1: High-Pressure Work Cycle
        if weekday < 5: # Weekdays (Mon-Fri)
            val = np.random.uniform(0.60, 0.75) # High-ish but not critical
        else: # Weekends (Sat-Sun)
            val = np.random.uniform(0.15, 0.30) # Recovery
            
    elif 15 <= day <= 21:
        # Phase 2: Deadline Week (Consistent High)
        val = np.random.uniform(0.80, 0.95)
        
    else:
        # Phase 3: Post-Deadline Fatigue (The Burnout Wall)
        val = np.random.uniform(0.70, 0.80)
        
    stress_levels.append(round(val, 3))

# 3. Create DataFrame
mock_data_2 = pd.DataFrame({
    'day': days,
    'predicted_stress': stress_levels
})

print(mock_data_2.head(15))

    day  predicted_stress
0     1             0.681
1     2             0.614
2     3             0.663
3     4             0.693
4     5             0.633
5     6             0.285
6     7             0.266
7     8             0.653
8     9             0.721
9    10             0.640
10   11             0.628
11   12             0.703
12   13             0.224
13   14             0.217
14   15             0.865


# Prediction of Burnout Risk Over Time

## Single Exponential Decay Moving Average Model

In [14]:
def calculate_single_ema(new_prediction, prev_ema=None):
    # Alpha for a 7-day window
    alpha = 0.25 
    
    if prev_ema is None:
        return new_prediction
    
    # Standard EMA Formula
    current_ema = (new_prediction * alpha) + (prev_ema * (1 - alpha))
    return round(current_ema, 3)

# The "Basic" Trigger
def get_basic_burnout_tier(ema_score):
    if ema_score > threshold_dict["HIGH"]:
        return "HIGH", "Immediate Intervention"
    elif ema_score > threshold_dict["MEDIUM"]:
        return "MEDIUM", "Early Warning"
    else:
        return "LOW", "Stable"

In [17]:
for index, row in mock_data_1.iterrows():
    day = row['day']
    new_pred = row['predicted_stress']
    
    
    if index == 0:
        ema_score = calculate_single_ema(new_pred)
    else:
        ema_score = calculate_single_ema(new_pred, prev_ema)
    
    prev_ema = ema_score  # Update for next iteration
    
    tier, action = get_basic_burnout_tier(ema_score)
    
    print(f"Day {int(day):02d}: Predicted Stress={new_pred:.3f} (), EMA={ema_score:.3f} => Burnout Tier: {tier} | Action: {action}")

Day 01: Predicted Stress=0.030 (), EMA=0.030 => Burnout Tier: LOW | Action: Stable
Day 02: Predicted Stress=0.055 (), EMA=0.036 => Burnout Tier: LOW | Action: Stable
Day 03: Predicted Stress=0.060 (), EMA=0.042 => Burnout Tier: LOW | Action: Stable
Day 04: Predicted Stress=0.008 (), EMA=0.034 => Burnout Tier: LOW | Action: Stable
Day 05: Predicted Stress=0.042 (), EMA=0.036 => Burnout Tier: LOW | Action: Stable
Day 06: Predicted Stress=0.083 (), EMA=0.048 => Burnout Tier: LOW | Action: Stable
Day 07: Predicted Stress=0.009 (), EMA=0.038 => Burnout Tier: LOW | Action: Stable
Day 08: Predicted Stress=0.024 (), EMA=0.034 => Burnout Tier: LOW | Action: Stable
Day 09: Predicted Stress=0.080 (), EMA=0.046 => Burnout Tier: LOW | Action: Stable
Day 10: Predicted Stress=0.075 (), EMA=0.053 => Burnout Tier: LOW | Action: Stable
Day 11: Predicted Stress=0.023 (), EMA=0.046 => Burnout Tier: LOW | Action: Stable
Day 12: Predicted Stress=0.078 (), EMA=0.054 => Burnout Tier: LOW | Action: Stable
Day 